# AuxiliaryASR CTC Entropy Evaluation

This notebook loads a trained AuxiliaryASR model, prepares the validation dataset, and computes summary statistics of the CTC output distribution.

In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            'type': 'CUDA',
            'available': True,
            'name': device_name,
            'index': i
        })
else:
    devices.append({'type': 'CUDA', 'available': False, 'name': 'N/A'})
devices

In [ ]:
# change folder into the root of the ASR project
import os

if not os.path.isdir('Configs'):
    %cd ../

!pwd

## Imports and helper utilities

In [ ]:
# import packages, define helper utilities
import os
import yaml
import torch
import pandas as pd

from models import ASRCNN
from meldataset import build_dataloader
from token_map import build_token_map_from_data


def cfg_get_nested(cfg: dict, path, default=None, sep='.'):
    '''Get a nested value from a dict using a list of keys or a dot-separated string.'''
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur



def load_token_map_from_config(config):
    token_src = config.get('phoneme_maps_path')
    if not token_src:
        return build_token_map_from_data(
            config.get('train_data'),
            config.get('val_data'),
            config.get('ood_data'),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {word: index for word, index in csv}



def load_asr_model(model_path, config_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    token_map = load_token_map_from_config(config)

    model_params = cfg_get_nested(config, 'model_params', {
        'input_dim': 80,
        'hidden_dim': 256,
        'n_token': len(token_map),
        'token_embedding_dim': 512,
        'n_layers': 5,
        'location_kernel_size': 31,
    })
    if 'n_token' not in model_params:
        model_params['n_token'] = len(token_map)

    model = ASRCNN(**model_params)
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('model', checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized_state = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized_state)

    model.to(device)
    model.eval()
    return model, config, token_map



def build_dev_dataloader(config, device, batch_size=None, num_workers=None):
    val_csv_path = config.get('val_data')
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    with open(val_csv_path, 'r', encoding='utf-8') as f:
        raw_lines = [line.rstrip('
') for line in f]

    path_list = []
    for raw in raw_lines:
        if not raw.strip():
            continue
        parts = raw.split('|')
        if len(parts) == 1:
            continue
        path = parts[0]
        if len(parts) == 2:
            text = parts[1]
            speaker = ''
        else:
            text = '|'.join(parts[1:-1])
            speaker = parts[-1]
        path_list.append([path, text, speaker])

    if batch_size is None:
        batch_size = int(cfg_get_nested(config, 'eval_params.batch_size', cfg_get_nested(config, 'batch_size', 4)))
    if num_workers is None:
        num_workers = int(cfg_get_nested(config, 'dataloader_params.val_num_workers', 2))

    dataset_params = {
        'dict_path': cfg_get_nested(config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested(config, 'preprocess_params.sr', 24000),
        'spect_params': cfg_get_nested(
            config,
            'preprocess_params.spect_params',
            {'n_fft': 1024, 'win_length': 1024, 'hop_length': 300},
        ),
        'mel_params': cfg_get_nested(
            config, 'preprocess_params.mel_params', {'n_mels': 80}
        ),
    }

    collate_config = {'return_wave': False}
    device_flag = device.type if isinstance(device, torch.device) else str(device)
    loader = build_dataloader(
        path_list=path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device_flag,
        collate_config=collate_config,
        dataset_config=dataset_params,
    )
    return loader

## Load model, configuration, and validation loader

In [ ]:
checkpoint_dir = 'Checkpoint'
config_path = './Configs/config.yml'

if not os.path.isdir(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint directory '{checkpoint_dir}' not found.")

ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('epoch_') and f.endswith('.pth')]
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found in '{checkpoint_dir}'.")

ckpt_files = sorted(ckpt_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
model_path = os.path.join(checkpoint_dir, ckpt_files[-1])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config, token_map = load_asr_model(model_path, config_path, device)

print(f'model --> {model_path}')
print(f'config --> {config_path}')
phoneme_source = config.get('phoneme_maps_path', 'built from dataset')
print(f'dictionary --> {phoneme_source}')

dev_loader = build_dev_dataloader(config, device)
print(f'Validation dataset size: {len(dev_loader.dataset)} samples')

if ' ' not in token_map:
    raise KeyError("The vocabulary does not contain the blank symbol ' '.")
BLANK_ID = token_map[' ']
print(f'Blank token id: {BLANK_ID}')

## CTC entropy statistics

In [ ]:
# entropy statistics computation
import math

@torch.no_grad()
def ctc_entropy_stats(model, dev_loader, device=None, blank_id=0):
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    entropies, maxp, blank_rates = [], [], []
    downsample_factor = 2 ** getattr(model, 'n_down', 1)

    for batch in dev_loader:
        texts, text_lens, mels, mel_lens = batch[:4]
        mels = mels.to(device)
        mel_lens = mel_lens.to(torch.long)

        outputs = model(mels)
        if isinstance(outputs, dict):
            logits = outputs.get('logits_ctc')
            if logits is None:
                raise KeyError("Model output dict does not contain 'logits_ctc'.")
        elif isinstance(outputs, (tuple, list)):
            logits = outputs[0]
        else:
            logits = outputs

        logp = logits.log_softmax(-1).cpu()
        probs = logp.exp()
        entropy = -(probs * logp).sum(-1)
        max_probs, ids = probs.max(-1)
        blank_mask = ids.eq(blank_id)

        logit_lens = torch.clamp(mel_lens // downsample_factor, min=1, max=logits.size(1)).cpu()

        for h, m, bmask, L in zip(entropy, max_probs, blank_mask, logit_lens):
            length = int(L)
            entropies.append(h[:length].mean().item())
            maxp.append(m[:length].mean().item())
            blank_rates.append(bmask[:length].float().mean().item())

    return {
        'mean_entropy': float(torch.tensor(entropies).mean()),
        'mean_maxprob': float(torch.tensor(maxp).mean()),
        'mean_blank_rate': float(torch.tensor(blank_rates).mean()),
    }


stats = ctc_entropy_stats(model, dev_loader, device=device, blank_id=BLANK_ID)
print(stats)